# 13 — Factual Gold reconstruction

Uses only epoch-scoped Silver evidence and the existing backend `CardEvidenceProjection`. Latest, Peak, Lap, Gap, Interval and corrected valid Best Lap are reconstructed. Start requires an observed pre-start transition; Final/Change stay NULL because finished status is not approved classification evidence.

Names, photos, display-name joins and modelled/pass metrics are absent from these schemas. Every mutable identity includes logical session, source epoch and participant. No old mixed card-feed computation executes here.


In [ ]:
# Resolve shared code from the existing notebook in this kernel.
import os
import sys
import json
from pathlib import Path
roots = [Path(os.environ["AIDP_PROJECT_ROOT"])] if os.getenv("AIDP_PROJECT_ROOT") else [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "notebooks_aidp/10_event_capture_foundation.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
os.environ["AIDP_PROJECT_ROOT"] = str(root.resolve())
foundation = json.loads((root / "notebooks_aidp/10_event_capture_foundation.ipynb").read_text())
for cell in foundation["cells"]:
    if "aidp_shared" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "10_event_capture_foundation", "exec"), globals())
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
events = read_events(spark)
outputs = gold_rows(events)
preflight(spark, {table(GOLD_EVENT_SCHEMA, name): ddl for name, ddl in GOLD_TABLES.items()})
for name, rows in outputs.items():
    target = table(GOLD_EVENT_SCHEMA, name)
    assert_private_free(spark.table(target).columns)
    keys = ["logical_session_id", "source_epoch"]
    if name != "session_summary":
        keys.append("participant_id")
    if name == "participant_laps":
        keys.append("lap_number")
    merge_rows(spark, target, rows, GOLD_TABLES[name], keys)
    equal_rows(spark, target, rows, GOLD_TABLES[name], keys)
cards = spark.table(table(GOLD_EVENT_SCHEMA, "baseball_card_evidence"))
require(not cards.filter("upper(kart_number) = 'SC' OR final_position IS NOT NULL OR position_change IS NOT NULL").limit(1).count(),
        "SC/final classification privacy/evidence gate failed")
print({"factual_cards": cards.count(), "modelled_metrics": "UNSUPPORTED", "final_classification": "UNSUPPORTED"})
